# Lab 4 - Session 1
## Fork the Repository, Train YOLO26n, Evaluate, Export, and Push to the Group Fork

**Duration:** 2 hours  
**Session 1 marks:** 60 points  
**Official deliverable:** `SecXX_GroupXX_Lab4_Session1.pdf`

Complete all required cells, result tables, and questions. Keep all required outputs visible before printing the notebook to PDF.

### Mark distribution

- 27 questions
- 6 analytical questions × 3 marks = 18 marks
- 21 focused questions × 2 marks = 42 marks
- Total = 60 marks

Workflow:

`instructor upstream → group fork → Colab clone → train/evaluate/export → push student_work/ → Raspberry Pi clone in Session 2`


In [ ]:
#@title Student, group, and fork information
GROUP_MEMBER_NAMES = "" #@param {type:"string"}
SECTION_NUMBER = 1 #@param {type:"integer"}
GROUP_NUMBER = 1 #@param {type:"integer"}
FORK_OWNER = "" #@param {type:"string"}
FORK_REPOSITORY_NAME = "lab4-megalopa" #@param {type:"string"}
GITHUB_USERNAME_FOR_PUSH = "" #@param {type:"string"}
GIT_EMAIL = "" #@param {type:"string"}

assert GROUP_MEMBER_NAMES.strip(), "Enter all group-member names."
assert SECTION_NUMBER in (1, 2), "SECTION_NUMBER must be 1 or 2."
assert 1 <= GROUP_NUMBER <= 25, "GROUP_NUMBER must be between 1 and 25."
assert FORK_OWNER.strip(), "Enter the GitHub account that owns the group fork."
assert FORK_REPOSITORY_NAME.strip(), "Enter the fork repository name."
assert GITHUB_USERNAME_FOR_PUSH.strip(), "Enter the GitHub account used for the final push."
assert GIT_EMAIL.strip(), "Enter the email used for the Git commit."

UPSTREAM_URL = "https://github.com/WUStuLab/lab4-megalopa.git"
FORK_REPO_URL = f"https://github.com/{FORK_OWNER}/{FORK_REPOSITORY_NAME}.git"
REPO_DIR = Path("/content/lab4-megalopa") if "Path" in globals() else None

print("Group members:", GROUP_MEMBER_NAMES)
print("Section:", SECTION_NUMBER)
print("Group:", GROUP_NUMBER)
print("Group fork:", FORK_REPO_URL)
print("GitHub account used for push:", GITHUB_USERNAME_FOR_PUSH)


## 1. Core workflow

The instructor repository is the read-only upstream source for students. Each group works in its own fork.

### Questions

**Q1.** Explain the difference between model training and model inference. **[3 marks]**  
**Answer:** TYPE HERE

**Q2.** Why is training performed in Google Colab while deployment is performed on Raspberry Pi 5? **[3 marks]**  
**Answer:** TYPE HERE


## 2. Colab environment setup

Select a GPU runtime:

`Runtime → Change runtime type → T4 GPU`

Use a fresh runtime. Do not force-upgrade Colab's preinstalled PyTorch, NumPy, pip, or setuptools.


In [ ]:
%pip install -q ultralytics onnx onnxruntime onnxslim scikit-learn jedi


In [ ]:
import json
import math
import os
import random
import re
import shutil
import subprocess
import sys
import time
from pathlib import Path
from secrets import SystemRandom

import cv2
import matplotlib.pyplot as plt
import numpy as np
import onnx
import onnxruntime
import pandas as pd
import requests
import sklearn
import torch
import ultralytics
import yaml
from IPython.display import Video, display
from PIL import Image
from sklearn.model_selection import train_test_split
from ultralytics import YOLO

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

REPO_DIR = Path("/content/lab4-megalopa")
RUNTIME_DIR = Path("/content/lab4_runtime") / f"sec{SECTION_NUMBER:02d}_group{GROUP_NUMBER:02d}"
WORK_REL = Path("student_work")

summary = {
    "Python": sys.version.split()[0],
    "PyTorch": torch.__version__,
    "Ultralytics": ultralytics.__version__,
    "ONNX": onnx.__version__,
    "ONNX Runtime": onnxruntime.__version__,
    "Scikit-learn": sklearn.__version__,
    "GPU available": torch.cuda.is_available(),
    "GPU": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU runtime",
}
display(pd.DataFrame(summary.items(), columns=["Item", "Result"]))


### Environment evidence

Confirm the version table and GPU information remain visible in the final PDF.


## 3. Create the group fork before running the next cell

One designated group member must:

1. Open `https://github.com/WUStuLab/lab4-megalopa`.
2. Select **Fork**.
3. Create the fork under the designated student's GitHub account.
4. Add the remaining group members as collaborators to the fork.
5. Enter the fork owner and repository name in the form above.

The upstream repository does not need to grant students Write access.


## 4. Clone the public group fork

Cloning and pulling a public repository do not require authentication. Authentication is required only when the group pushes results back to its fork.


In [ ]:
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

subprocess.run([
    "git", "clone", "--depth", "1",
    FORK_REPO_URL,
    str(REPO_DIR),
], check=True)

os.chdir(REPO_DIR)

# Register the instructor repository as upstream for traceability.
existing_remotes = subprocess.check_output(["git", "remote"], text=True).split()
if "upstream" not in existing_remotes:
    subprocess.run(["git", "remote", "add", "upstream", UPSTREAM_URL], check=True)

current_branch = subprocess.check_output(
    ["git", "branch", "--show-current"], text=True
).strip()
latest_commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], text=True
).strip()

assert current_branch, "The fork does not have a checked-out default branch."
assert Path("shared").is_dir(), "The shared folder is missing from the fork."
assert WORK_REL.is_dir(), "The student_work folder is missing from the fork."

print("Fork cloned successfully.")
print("Origin:", subprocess.check_output(["git", "remote", "get-url", "origin"], text=True).strip())
print("Upstream:", subprocess.check_output(["git", "remote", "get-url", "upstream"], text=True).strip())
print("Current branch:", current_branch)
print("Latest commit:", latest_commit)
print("Top-level folders:", sorted(path.name for path in Path(".").iterdir() if not path.name.startswith(".")))


### Question

**Q3.** What is the difference between the instructor upstream repository and the group's fork? **[2 marks]**  
**Answer:** TYPE HERE


## 5. Restrict commits to `student_work/`

Students may read `shared/`, but their model and result commit must contain only files under `student_work/`.


In [ ]:
subprocess.run(["git", "config", "user.name", GITHUB_USERNAME_FOR_PUSH], check=True)
subprocess.run(["git", "config", "user.email", GIT_EMAIL], check=True)

hook_path = Path(".git/hooks/pre-commit")
hook_path.write_text(
    """#!/usr/bin/env bash
set -e
INVALID=0
while IFS= read -r FILE; do
  [[ -z \"$FILE\" ]] && continue
  if [[ \"$FILE\" != student_work/* ]]; then
    echo \"ERROR: Only student_work/ may be committed. Invalid path: $FILE\"
    INVALID=1
  fi
done < <(git diff --cached --name-only --diff-filter=ACMRD)
exit $INVALID
""",
    encoding="utf-8",
)
hook_path.chmod(0o755)

print("Git identity configured.")
print("Pre-commit guard installed:", hook_path)


### Question

**Q4.** Why must students stage only `student_work/` instead of using `git add .`? **[2 marks]**  
**Answer:** TYPE HERE


## 6. Inspect the shared YOLO box-label dataset

The notebook reports the actual number of files, source-image resolution, class IDs, and labelled-object count.

For a megalopa-only dataset, use:

```python
CLASS_NAMES = {0: "Crablet1", 1:"debris"}
```

If the dataset deliberately contains more classes, the instructor must revise this mapping before students begin.


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
CLASS_NAMES = {0: "Crablet1", 1:"debris"} # Instructor must verify this mapping.

images_dir = Path("shared/dataset/images")
labels_dir = Path("shared/dataset/labels")

image_paths = sorted(path for path in images_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)
label_paths = sorted(labels_dir.glob("*.txt"))

assert image_paths, f"No images found in {images_dir}"
assert label_paths, f"No labels found in {labels_dir}"

image_stems = {path.stem for path in image_paths}
label_stems = {path.stem for path in label_paths}
missing_labels = sorted(image_stems - label_stems)
orphan_labels = sorted(label_stems - image_stems)
if missing_labels or orphan_labels:
    raise ValueError({
        "images_without_labels": missing_labels[:20],
        "labels_without_images": orphan_labels[:20],
    })

sizes = []
class_counts = {}
object_total = 0
invalid_entries = []
unexpected_classes = []

for image_path in image_paths:
    with Image.open(image_path) as image:
        sizes.append(image.size)

    label_path = labels_dir / f"{image_path.stem}.txt"
    for line_number, line in enumerate(label_path.read_text(encoding="utf-8").splitlines(), start=1):
        if not line.strip():
            continue
        fields = line.split()
        if len(fields) != 5:
            invalid_entries.append(f"{label_path}:{line_number} has {len(fields)} fields")
            continue
        class_id = int(float(fields[0]))
        coordinates = list(map(float, fields[1:]))
        if not all(0.0 <= value <= 1.0 for value in coordinates):
            invalid_entries.append(f"{label_path}:{line_number} has coordinates outside [0, 1]")
        if class_id not in CLASS_NAMES:
            unexpected_classes.append(f"{label_path}:{line_number} uses class ID {class_id}")
        class_counts[class_id] = class_counts.get(class_id, 0) + 1
        object_total += 1

report = pd.DataFrame([
    ["Images", len(image_paths)],
    ["Label files", len(label_paths)],
    ["Source-image resolutions", ", ".join(map(str, sorted(set(sizes))))],
    ["Configured classes", CLASS_NAMES],
    ["Observed class counts", class_counts],
    ["Total labelled objects", object_total],
], columns=["Item", "Result"])
display(report)

if invalid_entries:
    print("Invalid label examples:")
    for item in invalid_entries[:20]:
        print(" -", item)
    raise ValueError(f"Found {len(invalid_entries)} invalid YOLO entries.")

if unexpected_classes:
    print("Unexpected class examples:")
    for item in unexpected_classes[:20]:
        print(" -", item)
    raise ValueError(
        f"Dataset contains class IDs outside CLASS_NAMES={CLASS_NAMES}. "
        "Remove/remap the labels or update the intended class mapping before training."
    )

print("Dataset validation passed.")


In [ ]:
from collections import Counter
from matplotlib.patches import Rectangle
from secrets import SystemRandom

samples = SystemRandom().sample(image_paths, min(5, len(image_paths)))
colors = {i: plt.cm.tab10(i % 10) for i in CLASS_NAMES}
fig, axes = plt.subplots(len(samples), 2, figsize=(12, 4.5 * len(samples)), squeeze=False)
summary = []

for r, img_path in enumerate(samples):
    img = np.array(Image.open(img_path).convert("RGB"))
    h, w = img.shape[:2]
    labels = [list(map(float, x.split())) for x in
              (labels_dir / f"{img_path.stem}.txt").read_text().splitlines() if x.strip()]
    counts = Counter(CLASS_NAMES.get(int(x[0]), f"class_{int(x[0])}") for x in labels)
    name = img_path.name if len(img_path.name) <= 38 else f"{img_path.name[:17]}...{img_path.name[-17:]}"

    axes[r, 0].imshow(img)
    axes[r, 0].set_title(f"Original\n{name}\n{w}×{h}", fontsize=8)
    axes[r, 0].axis("off")
    axes[r, 1].imshow(img)

    for n, (cid, xc, yc, bw, bh) in enumerate(labels, 1):
        cid, x, y = int(cid), (xc - bw / 2) * w, (yc - bh / 2) * h
        label, color = CLASS_NAMES.get(cid, f"class_{cid}"), colors.get(cid, "black")
        axes[r, 1].add_patch(Rectangle((x, y), bw*w, bh*h, fill=False,
                                       edgecolor=color, linewidth=1.8))
        axes[r, 1].text(x, max(0, y-2), f"{n}: {label}", fontsize=6, color="white",
                        bbox=dict(facecolor=color, alpha=.8, pad=1, edgecolor="none"))

    class_text = ", ".join(f"{k}: {v}" for k, v in counts.items()) or "None"
    axes[r, 1].set_title(f"YOLO labels\n{name}\nObjects: {len(labels)} | {class_text}",
                         fontsize=8)
    axes[r, 1].axis("off")
    summary.append({"Image": name, "Resolution": f"{w}×{h}",
                    "Objects": len(labels), "Classes": class_text})

plt.tight_layout()
plt.show()
display(pd.DataFrame(summary))
print("Total objects:", sum(x["Objects"] for x in summary))
print("Rerun for another random set.")

In [ ]:
display(pd.DataFrame({
    "Item": [
        "Total images", "Total label files", "Source resolution",
        "Configured classes", "Class name(s)", "Labelled objects"
    ],
    "Result": [
        len(image_paths), len(label_paths),
        ", ".join(f"{w}×{h}" for w, h in sorted(set(sizes))),
        len(CLASS_NAMES), ", ".join(CLASS_NAMES.values()), object_total
    ]
}))

### Dataset result

| Item | Result |
|---|---|
| Total images | TYPE HERE |
| Total label files | TYPE HERE |
| Source-image resolution | TYPE HERE |
| Number of configured classes | TYPE HERE |
| Class name(s) | TYPE HERE |
| Total labelled objects | TYPE HERE |

### Questions

**Q5.** What information is stored in one YOLO bounding-box label line? **[2 marks]**  
**Answer:** TYPE HERE

**Q6.** How can missing ground-truth boxes affect model training and evaluation? **[2 marks]**  
**Answer:** TYPE HERE

**Q7.** How can boxes placed on debris or reflections affect the detector? **[2 marks]**  
**Answer:** TYPE HERE

**Q8.** State one benefit and one limitation of using a training input size of 640 × 640 for small-object detection. **[2 marks]**  
**Answer:** TYPE HERE


## 7. Create a 70% / 20% / 10% split outside the repository

The split is generated in temporary Colab storage to avoid duplicating the shared dataset in every fork.


In [ ]:
if RUNTIME_DIR.exists():
    shutil.rmtree(RUNTIME_DIR)
RUNTIME_DIR.mkdir(parents=True)
split_root = RUNTIME_DIR / "dataset_split"

assert len(image_paths) >= 10, "At least 10 images are required."
train_paths, remaining_paths = train_test_split(image_paths, test_size=0.30, random_state=SEED, shuffle=True)
valid_paths, test_paths = train_test_split(remaining_paths, test_size=1/3, random_state=SEED, shuffle=True)
subsets = {"train": train_paths, "valid": valid_paths, "test": test_paths}

for subset_name, subset_paths in subsets.items():
    image_output = split_root / subset_name / "images"
    label_output = split_root / subset_name / "labels"
    image_output.mkdir(parents=True, exist_ok=True)
    label_output.mkdir(parents=True, exist_ok=True)
    for image_path in subset_paths:
        shutil.copy2(image_path, image_output / image_path.name)
        label_path = labels_dir / f"{image_path.stem}.txt"
        shutil.copy2(label_path, label_output / label_path.name)

data_yaml = {
    "path": str(split_root.resolve()),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": CLASS_NAMES,
}
(split_root / "data.yaml").write_text(yaml.safe_dump(data_yaml, sort_keys=False), encoding="utf-8")

display(pd.DataFrame([
    ["Training", "70%", len(train_paths)],
    ["Validation", "20%", len(valid_paths)],
    ["Test", "10%", len(test_paths)],
    ["Total", "100%", len(image_paths)],
], columns=["Subset", "Target proportion", "Actual images"]))
print((split_root / "data.yaml").read_text(encoding="utf-8"))


### Questions

**Q9.** What is the role of the training set? **[2 marks]**  
**Answer:** TYPE HERE

**Q10.** What is the role of the validation set? **[2 marks]**  
**Answer:** TYPE HERE

**Q11.** What is the role of the test set? **[2 marks]**  
**Answer:** TYPE HERE

**Q12.** What is data leakage, and why are near-duplicate video frames a leakage risk? **[3 marks]**  
**Answer:** TYPE HERE


## 8. Configure YOLO26n training


In [ ]:
#@title Training configuration
EPOCHS = 15 #@param {type:"integer"}
IMAGE_SIZE = 640 #@param {type:"integer"}
PATIENCE = 10 #@param {type:"integer"}

BATCH_SIZE = 16 if torch.cuda.is_available() else 4
DEVICE = 0 if torch.cuda.is_available() else "cpu"
RUN_NAME = f"sec{SECTION_NUMBER:02d}_group{GROUP_NUMBER:02d}_yolo26n_megalopa"
TRAIN_RUNTIME = RUNTIME_DIR / "training"

local_base_model = Path("shared/models/yolo26n.pt")
BASE_MODEL = str(local_base_model) if local_base_model.exists() else "yolo26n.pt"
model = YOLO(BASE_MODEL)

display(pd.DataFrame([
    ["Base model", BASE_MODEL],
    ["Epochs", EPOCHS],
    ["Training input size", IMAGE_SIZE],
    ["Batch size", BATCH_SIZE],
    ["Early-stopping patience", PATIENCE],
    ["Device", DEVICE],
], columns=["Parameter", "Value"]))


### Questions

**Q13.** What does one training epoch represent? **[2 marks]**  
**Answer:** TYPE HERE

**Q14.** What is batch size, and what can happen if it is too large for GPU memory? **[2 marks]**  
**Answer:** TYPE HERE

**Q15.** Why is a pretrained YOLO26n model used instead of starting from random weights? **[2 marks]**  
**Answer:** TYPE HERE

**Q16.** Explain the accuracy-speed trade-off associated with the training input image size. **[3 marks]**  
**Answer:** TYPE HERE


## 9. Train the model


In [ ]:
started = time.perf_counter()
train_results = model.train(
    data=str((split_root / "data.yaml").resolve()),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    patience=PATIENCE,
    pretrained=True,
    device=DEVICE,
    project=str(TRAIN_RUNTIME),
    name=RUN_NAME,
    seed=SEED,
    exist_ok=True,
    plots=True,
)
training_seconds = time.perf_counter() - started
run_dir = Path(train_results.save_dir)
best_runtime = run_dir / "weights" / "best.pt"
assert best_runtime.exists(), f"best.pt was not found: {best_runtime}"

display(pd.DataFrame([
    ["Epochs requested", EPOCHS],
    ["Batch size", BATCH_SIZE],
    ["Training input size", IMAGE_SIZE],
    ["Training time (minutes)", round(training_seconds / 60, 2)],
    ["Best model", best_runtime],
], columns=["Training item", "Result"]))


### Questions

**Q17.** What is the difference between training loss and validation loss? **[2 marks]**  
**Answer:** TYPE HERE

**Q18.** State one curve pattern that may indicate overfitting, and explain why. **[3 marks]**  
**Answer:** TYPE HERE


## 10. Evaluate YOLO metrics on the test set

The next cell displays the numerical metrics and the available YOLO figures, including confusion matrices and confidence curves.


In [ ]:
evaluation_root = RUNTIME_DIR / "evaluation"
if evaluation_root.exists():
    shutil.rmtree(evaluation_root)

best_model = YOLO(str(best_runtime))
metric_output = best_model.val(
    data=str((split_root / "data.yaml").resolve()),
    split="test",
    imgsz=IMAGE_SIZE,
    conf=0.001,
    device=DEVICE,
    project=str(evaluation_root),
    name="test_metrics",
    exist_ok=True,
    plots=True,
)
evaluation_dir = Path(metric_output.save_dir)
metrics_summary = {
    "Precision": float(metric_output.box.mp),
    "Recall": float(metric_output.box.mr),
    "mAP@50": float(metric_output.box.map50),
    "mAP@50-95": float(metric_output.box.map),
    "Training time (seconds)": training_seconds,
}
display(pd.DataFrame(metrics_summary.items(), columns=["Metric", "Result"]))

plot_candidates = [
    ("Training results", run_dir / "results.png"),
    ("Confusion matrix", evaluation_dir / "confusion_matrix.png"),
    ("Normalized confusion matrix", evaluation_dir / "confusion_matrix_normalized.png"),
    ("Precision-Recall curve", evaluation_dir / "PR_curve.png"),
    ("F1-confidence curve", evaluation_dir / "F1_curve.png"),
    ("Precision-confidence curve", evaluation_dir / "P_curve.png"),
    ("Recall-confidence curve", evaluation_dir / "R_curve.png"),
    ("Training-label distribution", run_dir / "labels.jpg"),
    ("Label correlogram", run_dir / "labels_correlogram.jpg"),
    ("Validation labels", run_dir / "val_batch0_labels.jpg"),
    ("Validation predictions", run_dir / "val_batch0_pred.jpg"),
]
available_plots = [(title, path) for title, path in plot_candidates if path.exists()]

if available_plots:
    columns = 2
    rows = math.ceil(len(available_plots) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(16, 6 * rows), squeeze=False)
    for axis in axes.flat:
        axis.axis("off")
    for axis, (title, path) in zip(axes.flat, available_plots):
        axis.imshow(Image.open(path))
        axis.set_title(title)
        axis.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("No evaluation plots were found.")

print("Evaluation directory:", evaluation_dir)


### Metric result

| Metric | Result |
|---|---:|
| Precision | TYPE HERE |
| Recall | TYPE HERE |
| mAP@50 | TYPE HERE |
| mAP@50-95 | TYPE HERE |

### Questions

**Q19.** What does precision measure in the megalopa detector? **[2 marks]**  
**Answer:** TYPE HERE

**Q20.** What does recall measure in the megalopa detector? **[2 marks]**  
**Answer:** TYPE HERE

**Q21.** Why is mAP@50-95 normally lower than mAP@50? **[2 marks]**  
**Answer:** TYPE HERE

**Q22.** Give one concrete false-positive example and one false-negative example for this dataset. **[2 marks]**  
**Answer:** TYPE HERE


## 11. Save `best.pt` and export ONNX into `student_work/`


In [ ]:
model_dir = WORK_REL / "models"
training_result_dir = WORK_REL / "results/training"
validation_result_dir = WORK_REL / "results/validation"
information_dir = WORK_REL / "session_information"
for directory in (model_dir, training_result_dir, validation_result_dir, information_dir):
    directory.mkdir(parents=True, exist_ok=True)

model_stem = f"sec{SECTION_NUMBER:02d}_group{GROUP_NUMBER:02d}_megalopa_yolo26n_best"
repo_pt = model_dir / f"{model_stem}.pt"
shutil.copy2(best_runtime, repo_pt)

for _, source_path in available_plots:
    shutil.copy2(source_path, training_result_dir / source_path.name)
(training_result_dir / "metrics.json").write_text(json.dumps(metrics_summary, indent=2), encoding="utf-8")

print("Saved PyTorch model:", repo_pt)
print("PyTorch model size (MB):", round(repo_pt.stat().st_size / 1024**2, 2))


In [ ]:
export_model = YOLO(str(repo_pt))
onnx_exported = Path(export_model.export(format="onnx", imgsz=IMAGE_SIZE, simplify=True))
repo_onnx = model_dir / f"{model_stem}.onnx"
if onnx_exported.resolve() != repo_onnx.resolve():
    shutil.copy2(onnx_exported, repo_onnx)

onnx_model = YOLO(str(repo_onnx), task="detect")
test_image = SystemRandom().choice(sorted((split_root / "test/images").glob("*")))
pt_result = YOLO(str(repo_pt)).predict(source=str(test_image), imgsz=IMAGE_SIZE, conf=0.25, verbose=False)[0]
onnx_result = onnx_model.predict(source=str(test_image), imgsz=IMAGE_SIZE, conf=0.25, verbose=False)[0]

display(pd.DataFrame([
    ["PyTorch", repo_pt.name, round(repo_pt.stat().st_size / 1024**2, 2), len(pt_result.boxes)],
    ["ONNX", repo_onnx.name, round(repo_onnx.stat().st_size / 1024**2, 2), len(onnx_result.boxes)],
], columns=["Format", "Filename", "Size (MB)", "Predicted count"]))
print("Comparison image:", test_image.name)


### Questions

**Q23.** Why is `best.pt` retained instead of using only `last.pt`? **[2 marks]**  
**Answer:** TYPE HERE

**Q24.** Does conversion from `.pt` to ONNX retrain the model or learn new weights? Explain. **[2 marks]**  
**Answer:** TYPE HERE

**Q25.** Why must the exported ONNX model be loaded and tested after conversion? **[2 marks]**  
**Answer:** TYPE HERE


### Optional: export NCNN

NCNN export is optional and is not included in the 60 marks.


In [ ]:
# OPTIONAL
# ncnn_exported = Path(export_model.export(format="ncnn", imgsz=IMAGE_SIZE))
# destination = model_dir / f"{model_stem}_ncnn_model"
# if destination.exists():
#     shutil.rmtree(destination)
# shutil.copytree(ncnn_exported, destination)
# print("NCNN model saved:", destination)


## 12. Validate the trained model using test images, supplied validation images, and the supplied video

Rerun the image-validation cell to select another random test sample.


In [ ]:
def annotate_and_save_images(image_list, output_directory, title_prefix, label_directory=None):
    output_directory.mkdir(parents=True, exist_ok=True)
    records = []
    rendered = []

    for image_path in image_list:
        result = best_model.predict(
            source=str(image_path), imgsz=IMAGE_SIZE, conf=0.25,
            device=DEVICE, verbose=False,
        )[0]
        annotated_bgr = result.plot()
        annotated_rgb = cv2.cvtColor(annotated_bgr, cv2.COLOR_BGR2RGB)
        output_path = output_directory / f"{image_path.stem}_detected.jpg"
        Image.fromarray(annotated_rgb).save(output_path)

        actual_count = None
        if label_directory is not None:
            label_path = label_directory / f"{image_path.stem}.txt"
            if label_path.exists():
                actual_count = sum(1 for line in label_path.read_text(encoding="utf-8").splitlines() if line.strip())

        predicted_count = len(result.boxes)
        records.append({
            "image": image_path.name,
            "actual_count": actual_count,
            "predicted_count": predicted_count,
            "saved_result": output_path.as_posix(),
        })
        rendered.append((
            f"{title_prefix}: {image_path.name}\nactual={actual_count if actual_count is not None else 'unknown'}, predicted={predicted_count}",
            annotated_rgb,
        ))

    columns = 2
    rows = math.ceil(len(rendered) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(14, 6 * rows), squeeze=False)
    for axis in axes.flat:
        axis.axis("off")
    for axis, (title, image_array) in zip(axes.flat, rendered):
        axis.imshow(image_array)
        axis.set_title(title)
        axis.axis("off")
    plt.tight_layout()
    plt.show()
    return pd.DataFrame(records)

# Five random labelled test images.
test_image_paths = sorted((split_root / "test/images").glob("*"))
selected_test_images = SystemRandom().sample(test_image_paths, k=min(5, len(test_image_paths)))
test_validation_table = annotate_and_save_images(
    selected_test_images,
    validation_result_dir / "test_images",
    "Test image",
    label_directory=split_root / "test/labels",
)
display(test_validation_table)
test_validation_table.to_csv(validation_result_dir / "test_image_results.csv", index=False)

# Up to five supplied validation images.
supplied_validation_dir = Path("shared/validation/images")
supplied_validation_images = sorted(path for path in supplied_validation_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)
if supplied_validation_images:
    selected_supplied = SystemRandom().sample(supplied_validation_images, k=min(5, len(supplied_validation_images)))
    supplied_table = annotate_and_save_images(
        selected_supplied,
        validation_result_dir / "supplied_images",
        "Supplied validation image",
    )
    display(supplied_table)
    supplied_table.to_csv(validation_result_dir / "supplied_image_results.csv", index=False)
else:
    print("No supplied validation images were found.")


In [ ]:
#@title Video-validation settings
VALIDATION_VIDEO_INDEX = 0 #@param {type:"integer"}
MAX_VIDEO_SECONDS = 30 #@param {type:"integer"}
VIDEO_CONFIDENCE = 0.25 #@param {type:"number"}

video_directory = Path("shared/validation/videos")
video_extensions = {".mp4", ".avi", ".mov", ".mkv", ".m4v"}
video_candidates = sorted(path for path in video_directory.iterdir() if path.suffix.lower() in video_extensions)
assert video_candidates, f"No validation video found in {video_directory}"
assert 0 <= VALIDATION_VIDEO_INDEX < len(video_candidates)
video_path = video_candidates[VALIDATION_VIDEO_INDEX]
print("Selected video:", video_path)

capture = cv2.VideoCapture(str(video_path))
if not capture.isOpened():
    raise RuntimeError(f"OpenCV could not open: {video_path}")
source_fps = capture.get(cv2.CAP_PROP_FPS)
source_fps = source_fps if source_fps and source_fps > 0 else 15.0
frame_width = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH))
frame_height = int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
source_frames = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
maximum_frames = int(source_fps * MAX_VIDEO_SECONDS) if MAX_VIDEO_SECONDS > 0 else source_frames

raw_video_path = RUNTIME_DIR / "validation_video_raw.mp4"
writer = cv2.VideoWriter(
    str(raw_video_path), cv2.VideoWriter_fourcc(*"mp4v"),
    source_fps, (frame_width, frame_height),
)
if not writer.isOpened():
    capture.release()
    raise RuntimeError("OpenCV could not create the output video.")

frame_count = 0
predicted_counts = []
video_started = time.perf_counter()
while frame_count < maximum_frames:
    success, frame = capture.read()
    if not success:
        break
    result = best_model.predict(
        source=frame, imgsz=IMAGE_SIZE, conf=VIDEO_CONFIDENCE,
        device=DEVICE, verbose=False,
    )[0]
    writer.write(result.plot())
    predicted_counts.append(len(result.boxes))
    frame_count += 1
capture.release()
writer.release()
processing_seconds = time.perf_counter() - video_started

output_video = validation_result_dir / f"{video_path.stem}_annotated_h264.mp4"
ffmpeg_result = subprocess.run([
    "ffmpeg", "-y", "-loglevel", "error",
    "-i", str(raw_video_path),
    "-c:v", "libx264", "-pix_fmt", "yuv420p", "-movflags", "+faststart",
    str(output_video),
], text=True, capture_output=True)
if ffmpeg_result.returncode != 0:
    print("H.264 conversion failed; using raw MP4 output.")
    print(ffmpeg_result.stderr[-1000:])
    output_video = validation_result_dir / f"{video_path.stem}_annotated.mp4"
    shutil.copy2(raw_video_path, output_video)

video_summary = {
    "source_video": video_path.name,
    "processed_frames": frame_count,
    "source_fps": source_fps,
    "processed_duration_seconds": frame_count / source_fps if source_fps else 0,
    "processing_seconds": processing_seconds,
    "average_processing_fps": frame_count / processing_seconds if processing_seconds else 0,
    "average_predicted_count": float(np.mean(predicted_counts)) if predicted_counts else 0,
    "confidence": VIDEO_CONFIDENCE,
    "output_video": output_video.as_posix(),
}
(validation_result_dir / "video_validation_summary.json").write_text(json.dumps(video_summary, indent=2), encoding="utf-8")
display(pd.DataFrame(video_summary.items(), columns=["Video item", "Result"]))
assert output_video.exists() and output_video.stat().st_size > 0
display(Video(str(output_video), embed=True, width=720, html_attributes="controls"))


### Validation observations

| Observation | Student entry |
|---|---|
| Correct test-image detection example | TYPE HERE |
| False positive or false negative example | TYPE HERE |
| Result on supplied validation images | TYPE HERE |
| Video detection stable between frames? | TYPE HERE |
| Main difficult visual condition | TYPE HERE |

**Q26.** Why can detections appear and disappear between consecutive video frames, and why is object detection not the same as object tracking? **[3 marks]**  
**Answer:** TYPE HERE


## 13. Commit and push only `student_work/` to the group fork

The fork is public, so cloning was anonymous. Pushing requires authentication.

Before running the final cell:

1. Create a fine-grained personal access token under the GitHub account used for the push.
2. Select only the group fork.
3. Set **profile > setting > Developer Settings> Personal access tokens > Fine-grained tokens > Generate new > Only select repositories > select repository
Repository permissions → Contents: Read and write**.

4. Store the token in Colab Secrets as `GITHUB_TOKEN`.
5. Enable notebook access to the secret.


**Open the Session 1 notebook in Google Colab.**

1. Look at the left sidebar.

2. Click the key icon to open Secrets.
3. Click Add new secret.
4. Enter:
Name: GITHUB_TOKEN
Value: github_pat_XXXXXXXXXXXXXXXXXXXXXXXX

In [ ]:
from google.colab import userdata
userdata.get('GITHUB_TOKEN')

In [ ]:
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")
print("Token available:", bool(token))

In [ ]:
import requests
from google.colab import userdata

token = userdata.get("GITHUB_TOKEN")

response = requests.get(
    "https://api.github.com/user",
    headers={
        "Authorization": f"Bearer {token}",
        "Accept": "application/vnd.github+json",
    },
    timeout=30,
)

if response.status_code == 200:
    print("Authenticated GitHub user:", response.json()["login"])
else:
    print("Authentication failed:", response.status_code)

In [ ]:
from google.colab import userdata

session_summary = {
    "group_members": GROUP_MEMBER_NAMES,
    "section": SECTION_NUMBER,
    "group": GROUP_NUMBER,
    "fork_url": FORK_REPO_URL,
    "github_username_for_push": GITHUB_USERNAME_FOR_PUSH,
    "model_pt": repo_pt.as_posix(),
    "model_onnx": repo_onnx.as_posix(),
    **metrics_summary,
}
(information_dir / "session1_summary.json").write_text(json.dumps(session_summary, indent=2), encoding="utf-8")

subprocess.run(["git", "add", WORK_REL.as_posix()], check=True)
staged_files = subprocess.check_output(["git", "diff", "--cached", "--name-only"], text=True).splitlines()
invalid = [path for path in staged_files if not path.startswith("student_work/")]
print("Staged files:")
for path in staged_files:
    print(" -", path)
if invalid:
    raise RuntimeError("Files outside student_work/ are staged:\n" + "\n".join(invalid))

if staged_files:
    subprocess.run([
        "git", "commit", "-m",
        f"Complete Session 1: Section {SECTION_NUMBER:02d} Group {GROUP_NUMBER:02d}",
    ], check=True)
else:
    print("No new files to commit.")

try:
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = None
if not GITHUB_TOKEN:
    raise RuntimeError("GITHUB_TOKEN was not found in Colab Secrets.")

# Verify the token can push to the fork before invoking Git.
headers = {
    "Authorization": f"Bearer {GITHUB_TOKEN}",
    "Accept": "application/vnd.github+json",
    "X-GitHub-Api-Version": "2022-11-28",
}
repo_response = requests.get(
    f"https://api.github.com/repos/{FORK_OWNER}/{FORK_REPOSITORY_NAME}",
    headers=headers,
    timeout=30,
)
if repo_response.status_code != 200:
    raise RuntimeError(
        f"Unable to verify fork permission ({repo_response.status_code}): "
        f"{repo_response.text[:400]}"
    )
permissions = repo_response.json().get("permissions", {})
print("Token repository permissions:", permissions)
if not permissions.get("push", False):
    raise PermissionError(
        "The token cannot push to the group fork. Confirm that the account owns or collaborates on the fork, "
        "and that the token has Contents: Read and write for this repository."
    )

askpass = Path("/content/lab4_git_askpass.sh")
askpass.write_text(
    "#!/usr/bin/env bash\n"
    'case "$1" in\n'
    '  *Username*) echo "x-access-token" ;;\n'
    '  *Password*) echo "$GITHUB_TOKEN" ;;\n'
    "esac\n",
    encoding="utf-8",
)
askpass.chmod(0o700)

git_env = os.environ.copy()
git_env["GITHUB_TOKEN"] = GITHUB_TOKEN
git_env["GIT_ASKPASS"] = str(askpass)
git_env["GIT_TERMINAL_PROMPT"] = "0"

push_result = subprocess.run(
    ["git", "push", "origin", "HEAD:main"],
    env=git_env,
    text=True,
    capture_output=True,
)
print(push_result.stdout)
print(push_result.stderr)
if push_result.returncode != 0:
    raise RuntimeError(
        "Git push failed. Confirm the token, collaborator access, and fork default branch.\n"
        + push_result.stderr[-2000:]
    )

final_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("Push completed.")
print("Fork URL:", FORK_REPO_URL.removesuffix(".git"))
print("Final Session 1 commit:", final_commit)
subprocess.run(["git", "status", "--short"], check=True)


### Git result

| Item | Result |
|---|---|
| Group members | TYPE HERE |
| Group fork URL | TYPE HERE |
| GitHub account used for push | TYPE HERE |
| Final Session 1 commit ID | TYPE HERE |
| `.pt` visible in `student_work/models/` | YES / NO |
| ONNX visible in `student_work/models/` | YES / NO |

**Q27.** Explain why the group fork URL and final Session 1 commit ID must be recorded for Session 2. **[2 marks]**  
**Answer:** TYPE HERE

## Session 1 PDF submission

1. Confirm all 27 answers are complete.
2. Confirm all required outputs are visible.
3. Use **File → Print → Save as PDF**.
4. Submit `SecXX_GroupXX_Lab4_Session1.pdf` through Google Classroom.
5. Do not display GitHub tokens or passwords in the PDF.
